In [0]:
%python
# ==============================================================================
# END-TO-END PIPELINE: BRONZE -> SILVER -> GOLD (MEDALLION ARCHITECTURE)
# STACK: AUTO LOADER + PYSPARK TRANSFORMATIONS + DELTA LAKE
# USE CASE: TALENT ASSET & RISK AUDIT GOVERNANCE
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- 1. ENVIRONMENT & PATH INITIALIZATION ---
spark.sql("CREATE CATALOG IF NOT EXISTS training_medallion")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.01_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.02_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.03_gold")

base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"
audit_landing = f"{base_volume}/streaming_talent_audit"
bronze_checkpoint = f"{base_volume}/meta_checkpoints/bronze_audit_stream"
bronze_schema = f"{base_volume}/meta_schemas/bronze_audit_stream"
silver_checkpoint = f"{base_volume}/meta_checkpoints/silver_audit_stream"

dbutils.fs.mkdirs(audit_landing)

# --- 2. PREPARE REFERENCE MASTER DATA (FOR RISK ENRICHMENT) ---
risk_ref_data = [
    ("FIN", "Finance & Treasury", 70.0, "Strict"),
    ("ENG", "Engineering", 85.0, "Moderate"),
    ("HR", "Human Resources", 60.0, "Strict"),
    ("OPS", "General Operations", 75.0, "Standard")
]
spark.createDataFrame(
    risk_ref_data, 
    ["department_code", "department_name", "max_risk_threshold", "compliance_level"]
).write.format("delta").mode("overwrite").saveAsTable("training_medallion.01_bronze.bronze_risk_parameters")


# --- 3. SIMULATE INCOMING RAW DATA (BRONZE SOURCE) ---
raw_payload = """{"audit_id": "AUD-101", "employee_id": 101, "dept_code": "FIN", "risk_score": 88, "flag_reason": "Excessive   Access  "}
{"audit_id": "AUD-102", "employee_id": 102, "dept_code": "HR", "risk_score": 25, "flag_reason": "Routine Check"}
{"audit_id": "AUD-103", "employee_id": 103, "dept_code": "ENG", "risk_score": -5, "flag_reason": null}
{"audit_id": "AUD-101", "employee_id": 101, "dept_code": "FIN", "risk_score": 88, "flag_reason": "Excessive Access"}"""

dbutils.fs.put(f"{audit_landing}/audit_events_batch1.json", raw_payload, overwrite=True)


# ==============================================================================
# STEP 1: INGESTION LAYER (LANDING -> BRONZE VIA AUTO LOADER)
# ==============================================================================
df_bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", bronze_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(audit_landing)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

query_bronze = (
    df_bronze_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .option("mergeSchema", "true")
    .table("training_medallion.01_bronze.bronze_talent_risk_audit")
)
query_bronze.awaitTermination()
print("✅ Bronze Ingestion Completed.")


# ==============================================================================
# STEP 2: TRANSFORMATION & GOVERNANCE LAYER (BRONZE -> SILVER)
# ==============================================================================
def process_silver_microbatch(batch_df, batch_id):
    """
    Transforms Bronze streaming micro-batch:
    Applies Cleansing, Deduplication, Casting, Enrichment, and Business Logic.
    """
    if batch_df.isEmpty():
        return

    # A. Data Cleansing & Filtering (Null Handling & Unwanted Spaces)
    df_cleaned = (
        batch_df
        .filter(F.col("audit_id").isNotNull() & F.col("employee_id").isNotNull())
        .withColumn("flag_reason", F.trim(F.regexp_replace(F.col("flag_reason"), " +", " ")))
        .withColumn("flag_reason", F.coalesce(F.col("flag_reason"), F.lit("Standard Verification")))
        .withColumn("risk_score", F.col("risk_score").cast("double"))
        .withColumn(
            "standardized_risk_score",
            F.when((F.col("risk_score").isNull()) | (F.col("risk_score") < 0), F.lit(0.0))
             .when(F.col("risk_score") > 100, F.lit(100.0))
             .otherwise(F.col("risk_score"))
        )
    )

    # B. Deduplication via Window Function
    window_dedup = Window.partitionBy("audit_id").orderBy(F.col("ingestion_timestamp").desc())
    df_deduped = (
        df_cleaned
        .withColumn("row_num", F.row_number().over(window_dedup))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    # C. Data Enrichment & Integration (Join with Risk Master Parameters)
    df_ref = spark.table("training_medallion.01_bronze.bronze_risk_parameters")
    
    df_enriched = (
        df_deduped.join(
            df_ref, 
            df_deduped.dept_code == df_ref.department_code, 
            how="left"
        )
        .na.fill({
            "department_name": "General Operations", 
            "max_risk_threshold": 75.0, 
            "compliance_level": "Standard"
        })
    )

    # D. Business Rules & Derived Columns (Talent Risk Evaluation)
    df_silver_final = (
        df_enriched
        .withColumn(
            "talent_risk_status",
            F.when(F.col("standardized_risk_score") >= F.col("max_risk_threshold"), F.lit("HIGH RISK EXCEEDED"))
             .when(F.col("standardized_risk_score") >= 50, F.lit("EVALUATION REQUIRED"))
             .otherwise(F.lit("COMPLIANT"))
        )
        .withColumn(
            "is_escalated",
            F.when(
                (F.col("talent_risk_status") == "HIGH RISK EXCEEDED") & (F.col("compliance_level") == "Strict"), 
                F.lit(True)
            ).otherwise(F.lit(False))
        )
        .withColumn("silver_processed_at", F.current_timestamp())
        .select(
            "audit_id", "employee_id", "dept_code", "department_name",
            "standardized_risk_score", "max_risk_threshold", "compliance_level",
            "talent_risk_status", "is_escalated", "flag_reason", 
            "ingestion_timestamp", "silver_processed_at"
        )
    )

    # Write incrementally to Silver Delta Table
    (
        df_silver_final.write
        .format("delta")
        .mode("append")
        .saveAsTable("training_medallion.02_silver.silver_talent_risk_audits")
    )

# Execute streaming pipeline from Bronze to Silver
df_bronze_reader = spark.readStream.table("training_medallion.01_bronze.bronze_talent_risk_audit")

query_silver = (
    df_bronze_reader.writeStream
    .trigger(availableNow=True)
    .option("checkpointLocation", silver_checkpoint)
    .foreachBatch(process_silver_microbatch)
    .start()
)
query_silver.awaitTermination()
print("✅ Silver Cleansing & Enrichment Completed.")


# ==============================================================================
# STEP 3: ANALYTICS & AGGREGATION LAYER (SILVER -> GOLD)
# ==============================================================================
# Gold Table: Aggregated talent risk metrics per department for executive KPI reporting
df_silver_table = spark.table("training_medallion.02_silver.silver_talent_risk_audits")

df_gold_kpi = (
    df_silver_table
    .groupBy("department_name", "compliance_level")
    .agg(
        F.count("audit_id").alias("total_audited_events"),
        F.round(F.avg("standardized_risk_score"), 2).alias("avg_dept_risk_score"),
        F.max("standardized_risk_score").alias("peak_risk_score"),
        F.sum(F.when(F.col("is_escalated") == True, 1).otherwise(0)).alias("total_escalations")
    )
    .withColumn("gold_updated_at", F.current_timestamp())
)

(
    df_gold_kpi.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("training_medallion.03_gold.gold_department_risk_summary")
)
print("✅ Gold KPI Aggregation Completed.")


# ==============================================================================
# STEP 4: VERIFY END-TO-END DATA FLOW ACROSS MEDALLION TIERS
# ==============================================================================
print("\n--- 1. BRONZE RAW DATA ---")
display(spark.table("training_medallion.01_bronze.bronze_talent_risk_audit"))

print("\n--- 2. SILVER ENRICHED DATA ---")
display(spark.table("training_medallion.02_silver.silver_talent_risk_audits"))

print("\n--- 3. GOLD ANALYTICS SUMMARY ---")
display(spark.table("training_medallion.03_gold.gold_department_risk_summary"))

Wrote 453 bytes.
✅ Bronze Ingestion Completed.
✅ Silver Cleansing & Enrichment Completed.
✅ Gold KPI Aggregation Completed.

--- 1. BRONZE RAW DATA ---


audit_status,employee_id,name,risk_score,_rescued_data,ingestion_time,source_file,department,training_completed,audit_id,dept_code,flag_reason,ingestion_timestamp
CRITICAL,null,Dewi Sartika,90,"{""employee_id"":""EMP_CORRUPT_99"",""_file_path"":""/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_3_corrupted.json""}",2026-10-25T15:51:07.330Z,/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_3_corrupted.json,IT,false,null,null,null,null
UNKNOWN,1005,Unknown Employee,null,"{""risk_score"":""INVALID_SCORE"",""_file_path"":""/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_3_corrupted.json""}",2026-10-25T15:51:07.330Z,/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_3_corrupted.json,null,null,null,null,null,null
null,101,null,88,null,null,/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_events_batch1.json,null,null,AUD-101,FIN,Excessive Access,2026-10-25T17:49:25.045Z
null,102,null,25,null,null,/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_events_batch1.json,null,null,AUD-102,HR,Routine Check,2026-10-25T17:49:25.045Z
null,103,null,-5,null,null,/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_events_batch1.json,null,null,AUD-103,ENG,null,2026-10-25T17:49:25.045Z
null,101,null,88,null,null,/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_events_batch1.json,null,null,AUD-101,FIN,Excessive Access,2026-10-25T17:49:25.045Z
null,1001,null,85,null,null,/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_stream_batch1.json,null,null,AUD-901,null,Excessive System Access,2026-10-25T17:49:25.045Z
null,1002,null,20,null,null,/Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_stream_batch1.json,null,null,AUD-902,null,Routine Check,2026-10-25T17:49:25.045Z
MEDIUM,1003,Rian Pratama,42,null,2026-10-25T15:49:25.530Z,/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_2_new_fields.json,Treasury,true,null,null,null,null
LOW,1001,Budi Santoso,15,null,2026-10-25T15:49:15.205Z,/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/audit_landing/batch_1_baseline.json,null,null,null,null,null,null



--- 2. SILVER ENRICHED DATA ---


audit_id,employee_id,dept_code,department_name,standardized_risk_score,max_risk_threshold,compliance_level,talent_risk_status,is_escalated,flag_reason,ingestion_timestamp,silver_processed_at
AUD-101,101,FIN,Finance & Treasury,88.0,70.0,Strict,HIGH RISK EXCEEDED,true,Excessive Access,2026-10-25T17:49:25.045Z,2026-10-25T17:49:56.555Z
AUD-102,102,HR,Human Resources,25.0,60.0,Strict,COMPLIANT,false,Routine Check,2026-10-25T17:49:25.045Z,2026-10-25T17:49:56.555Z
AUD-103,103,ENG,Engineering,0.0,85.0,Moderate,COMPLIANT,false,Standard Verification,2026-10-25T17:49:25.045Z,2026-10-25T17:49:56.555Z
AUD-901,1001,null,General Operations,85.0,75.0,Standard,HIGH RISK EXCEEDED,false,Excessive System Access,2026-10-25T17:49:25.045Z,2026-10-25T17:49:56.555Z
AUD-902,1002,null,General Operations,20.0,75.0,Standard,COMPLIANT,false,Routine Check,2026-10-25T17:49:25.045Z,2026-10-25T17:49:56.555Z



--- 3. GOLD ANALYTICS SUMMARY ---


department_name,compliance_level,total_audited_events,avg_dept_risk_score,peak_risk_score,total_escalations,gold_updated_at
Human Resources,Strict,1,25.0,25.0,0,2026-10-29T15:11:06.092Z
Finance & Treasury,Strict,1,88.0,88.0,1,2026-10-29T15:11:06.092Z
Engineering,Moderate,1,0.0,0.0,0,2026-10-29T15:11:06.092Z
General Operations,Standard,2,52.5,85.0,0,2026-10-29T15:11:06.092Z
